# CP-SAT Validation & Benchmark (Kaggle / Colab)

OR-Tools' CP-SAT solver hangs on macOS 15.x with ortools 9.15.x, so this notebook runs the validation on Linux (Kaggle / Colab) instead.

**What this notebook does:**
1. Install OR-Tools + repo dependencies
2. Clone the repo (or use a mounted path)
3. Sanity-check the CP-SAT solver on a trivial 1-var problem
4. Direct CP-SAT smoke test on a 5-pallet 40HC voyage
5. Run the full pytest suite for `test_cpsat.py`
6. Run `scripts/benchmark_cpsat.py` — CP-SAT vs 5 heuristics vs GA
7. Print the CSV summary

## 1. Install dependencies

In [ ]:
!pip install -q 'ortools>=9.10,<10' \
  'pydantic==2.9.2' 'pydantic-settings==2.6.1' \
  fastapi 'uvicorn[standard]' python-multipart websockets orjson loguru \
  numpy gymnasium deap shapely pandas \
  'pytest>=8' pytest-asyncio

## 2. Locate / clone the repo

Set `REPO_DIR` to whatever path the repo lives at on your environment. The block below tries common locations:
- Kaggle dataset: `/kaggle/input/loading-service-2`
- Colab git clone: `/content/loading-service-2`
- Otherwise: clone fresh from GitHub.

For a **private** repo, set `GITHUB_TOKEN` first (Kaggle: *Add-ons → Secrets*; Colab: `from google.colab import userdata; os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')`).

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = 'https://github.com/Seif-Sameh/loading-service-2.git'
candidates = [
    '/kaggle/input/loading-service-2',
    '/kaggle/working/loading-service-2',
    '/content/loading-service-2',
    './loading-service-2',
]
REPO_DIR = next((p for p in candidates if pathlib.Path(p).exists()), None)
if REPO_DIR is None:
    target = '/kaggle/working/loading-service-2' if pathlib.Path('/kaggle/working').exists() else './loading-service-2'
    tok = os.environ.get('GITHUB_TOKEN', '').strip()
    url = REPO_URL.replace('https://', f'https://{tok}@') if tok else REPO_URL
    subprocess.check_call(['git', 'clone', '--depth', '1', url, target])
    REPO_DIR = target
# If we landed on /kaggle/input/<dataset> (read-only), copy to /kaggle/working so pytest can write
if REPO_DIR.startswith('/kaggle/input/'):
    import shutil
    work = '/kaggle/working/loading-service-2'
    if not pathlib.Path(work).exists():
        shutil.copytree(REPO_DIR, work)
    REPO_DIR = work
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
print('contents:', sorted(os.listdir(REPO_DIR))[:20])

## 3. OR-Tools sanity — does CP-SAT even solve `x >= 5`?

Locally this hangs forever on macOS. On Linux this should finish in <0.1s.

In [ ]:
import time
from ortools.sat.python import cp_model
m = cp_model.CpModel()
x = m.NewIntVar(0, 10, 'x')
m.Add(x >= 5)
m.Maximize(x)
s = cp_model.CpSolver()
s.parameters.max_time_in_seconds = 5
t0 = time.perf_counter()
status = s.Solve(m)
print(f'status={s.StatusName(status)}  x={s.Value(x)}  in {time.perf_counter()-t0:.3f}s')
assert status == cp_model.OPTIMAL and s.Value(x) == 10, 'OR-Tools install broken'

## 4. Direct CP-SAT smoke test — 5 pallets in a 40HC

Calls `_solve_cpsat` directly with synthetic items. Bypasses the env/select replay so we can be certain the solver itself works before running the full benchmark.

In [ ]:
import os, sys, pathlib, time
# Self-contained: re-establish REPO_DIR after a possible kernel restart.
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms.cpsat import CPSATConfig, _solve_cpsat
from app.catalog.loader import get_container, get_cargo_preset

container = get_container('40HC')
items = [get_cargo_preset('eur_pallet_light', item_id=f'p{i}') for i in range(5)]
cfg = CPSATConfig(time_limit_s=20.0, num_search_workers=2, grid_mm=100, log_search_progress=True)
t0 = time.perf_counter()
placements, status, obj = _solve_cpsat(container, items, cfg)
print(f'\nstatus={status}  planned={len(placements)}/{len(items)}  obj={obj}  time={time.perf_counter()-t0:.2f}s')
for p in placements:
    print(f'  {p.item_id}: pos=({p.position.x_mm},{p.position.y_mm},{p.position.z_mm})  rot={p.rotation}')

## 5. Unit tests

In [ ]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
!cd {REPO_DIR} && python -m pytest tests/test_cpsat.py -v --tb=short 2>&1 | tail -40

## 6. Full benchmark — CP-SAT vs heuristics vs GA

Tweak `--voyages`, `--items`, `--cpsat-time` for budget. Defaults: 5 voyages × 30 items, 30s CP-SAT budget per voyage (≈2.5 min total for CP-SAT alone).

In [ ]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
!cd {REPO_DIR} && python -m scripts.benchmark_cpsat --voyages 5 --items 30 --cpsat-time 30 --cpsat-workers 4 --seed 42

### 6b. (Optional) Add MORL-PCT to the comparison

Set `MORL_CKPT` to the path of `morl_pct_latest.pt` (e.g., from a Kaggle dataset of trained models). Skips if not set.

In [ ]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
MORL_CKPT = ''  # e.g. '/kaggle/input/morl-pct-15m/morl_pct_latest.pt'
if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
    !pip install -q torch
    !cd {REPO_DIR} && python -m scripts.benchmark_cpsat --voyages 5 --items 30 --cpsat-time 30 --seed 42 --morl '{MORL_CKPT}'
else:
    print('Skipped: set MORL_CKPT to a real checkpoint path to include MORL-PCT.')

## 7. Show the CSV

In [ ]:
import pandas as pd, glob, os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
csvs = sorted(glob.glob(os.path.join(REPO_DIR, 'benchmarks/out/cpsat_benchmark_*.csv')))
if not csvs:
    print('No CSV found — did the benchmark run?')
else:
    df = pd.read_csv(csvs[-1])
    print('latest:', csvs[-1])
    summary = df.groupby('algorithm').agg(
        util_mean=('util_pct', 'mean'),
        util_std=('util_pct', 'std'),
        placed_mean=('placed_pct', 'mean'),
        access_eff=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
        time_s=('elapsed_s', 'mean'),
    ).round(2).sort_values('util_mean', ascending=False)
    print(summary)